In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud
from gtda.diagrams import PersistenceEntropy, PairwiseDistance
from gtda.diagrams import Silhouette
from gtda.diagrams import BettiCurve 
from gtda.homology import VietorisRipsPersistence
from gtda.time_series import TakensEmbedding, SlidingWindow
import plotly.express as px
from sklearn.metrics import mutual_info_score
from gtda.metaestimators import CollectionTransformer
from gtda.pipeline import Pipeline
from gtda.plotting import plot_heatmap

In [ ]:
slug_bd22 = pd.read_csv('processed_BD-22.csv')


In [ ]:
slug_bd22

In [ ]:
figure_BHP = px.scatter()
figure_BHP.add_scatter(x=slug_bd22['TimeStamp'], y=slug_bd22['BH_Pressure'],  name="BHP well22")
figure_BHP.show()

In [ ]:
slug_june = slug_bd22[:1000]
slug_june

In [ ]:
figure_BHP = px.scatter()
figure_BHP.add_scatter(x=slug_june['TimeStamp'], y=slug_june['BH_Pressure'],  name="BHP well22")
figure_BHP.add_scatter(x=slug_june['TimeStamp'], y=slug_june['TH_Pressure'],  name="BHP well22")

figure_BHP.show()

In [ ]:
from scipy.interpolate import CubicSpline

xmin = 1
xmax = len(slug_june['BH_Pressure'])-1
some_step = 0.1                         # this is a 6 seconds step, 10 points per minute
cs = CubicSpline(range(len(slug_june)), slug_june['BH_Pressure'])
x_range = np.arange(xmin, xmax, some_step)

slug_june.insert(1, "idx", np.linspace(0, len(slug_june)-1, num=len(slug_june)), True)

print(cs(x_range).shape)

# upsampled to 10 times

new_range = pd.date_range(slug_june['TimeStamp'].iloc[0], slug_june['TimeStamp'].iloc[-1], freq='6S')
new_range = new_range[0:len(cs(x_range))]

slug_spline = []
slug_spline = pd.DataFrame(data=cs(x_range)) 
slug_spline['TimeStamp'] = new_range
slug_spline

In [ ]:
slug_spline[:2401]

In [ ]:
from gtda.time_series import SlidingWindow, TakensEmbedding

signal = slug_spline[0]
print(len(signal))

window_size = 2400 
window_stride = 600
SW = SlidingWindow(size=window_size, stride=window_stride)

X_windows = SW.fit_transform(signal)
X_windows.shape

In [ ]:
def batch_analyzer(input_df, stride, max_embedding_dimension, max_time_delay):
    max_time_delay = int(max_time_delay)
    max_embedding_dimension = int(max_embedding_dimension)

    i = 0
    averages = np.zeros(shape=(len(input_df),2))
    
   # print('max time delay:',max_time_delay, 'max dim:',max_embedding_dimension)
    
    for timeserie in input_df:
        
        optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
            timeserie, max_time_delay, max_embedding_dimension, stride=stride
            )
        if optimal_embedding_dimension < 3:
            optimal_embedding_dimension = 3
            
        # print('analyzing nr.',i, 'dim',optimal_embedding_dimension,'delay', optimal_time_delay)
        averages[i] = [optimal_embedding_dimension, optimal_time_delay]
        i += 1

    print('Max Dimension:', averages.max(0))
    print('Mean Dimension:', averages.mean(0))
    return averages

In [ ]:
stats = batch_analyzer(X_windows, 1, 15, 45)
stats


In [ ]:
optimal_time_delay = 35
optimal_embedding_dimension = 12
stride = 3

from gtda.pipeline import Pipeline
from gtda.metaestimators import CollectionTransformer
from gtda.diagrams import PersistenceEntropy

TE = TakensEmbedding(time_delay=optimal_time_delay, dimension=optimal_embedding_dimension, stride=1)
homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
#PE = PersistenceEntropy()
#PE_norm = PersistenceEntropy(normalize=True)
#Betti = BettiCurve()

slugging_signals = X_windows
slugging_point_cloud  = TE.fit_transform(slugging_signals)
slugging_diagrams = VRP.fit_transform(slugging_point_cloud)

In [ ]:
pca = PCA(n_components=3)

target = 3
y_embedded_pca = pca.fit_transform(slugging_point_cloud[target])


plot_point_cloud(y_embedded_pca)

In [ ]:

max_time_delay = 60
max_embedding_dimension = 12
stride = 1

signal = X_windows[10][:1000]

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    signal, max_time_delay, max_embedding_dimension, stride=stride
    )
print('length of signal to analyze', int(len(signal)/stride))
print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay

embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_embedded = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_embedded_pca = pca.fit_transform(y_embedded)

plot_point_cloud(y_embedded_pca)

In [ ]:

homology_dimensions = (0,1,2,3)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram = VRP.fit_transform_plot(y_embedded[None,:,:])

In [ ]:
figure_BHP = px.scatter()
figure_BHP.add_scatter(y=X_windows[2],  name="BHP well22")
figure_BHP.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=2, cols=1)

fig.append_trace(go.Scatter(
    x=slug_spline['TimeStamp'],
    y=slug_spline[0],
    name="Bottom Hole P (Bar)"
), row=1, col=1)

fig.append_trace(go.Scatter(
    x=f,
    y=Pxx_den,
    name='Signal Spectrum'
), row=2, col=1)


fig.update_layout(height=600, width=1200, title_text="Slugging Event")
fig.show()

In [ ]:
print(1/0.00058)

In [ ]:
from statsmodels.tsa.seasonal import STL


results = STL(slug_spline[0], period=500).fit()
results.plot()
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

slugging_periodicity = 1000  # a guess of the oscillations periodicity
trending_time = 1000       # a guess of the time to trend 
test_timeseries = slug_spline[0]
result = seasonal_decompose(test_timeseries[:3000], model='additive', period=slugging_periodicity, extrapolate_trend=trending_time)
result.plot()
plt.show()